# extraction
 $\rightarrow$ **passage du spectre 2D $\rightarrow$ 1D (tracé, extraction, calibration, réponse instrumentale, normalisation)**



## import des libs

In [20]:
import numpy as np
import matplotlib.pyplot as plt

from astropy import units as u
from astropy.nddata import CCDData
import ccdproc as ccdproc
#from convenience_functions import show_image

import warnings, pathlib, os
from astropy.utils.exceptions import AstropyWarning, AstropyUserWarning
warnings.simplefilter('ignore', category=AstropyWarning)
warnings.simplefilter('ignore', category=AstropyUserWarning)
warnings.simplefilter('ignore', UserWarning)


In [21]:
from astropy.io import fits
from astropy.nddata import StdDevUncertainty
from astropy.coordinates import get_sun, AltAz, EarthLocation, SpectralCoord, SkyCoord
from astropy.time import Time
from astropy.convolution import Box1DKernel
from astropy.modeling import models, fitting

from specreduce.tracing import FlatTrace, FitTrace
from specreduce.background import Background
from specreduce.extract import BoxcarExtract
from specreduce.fluxcal import FluxCalibration
from specreduce.wavelength_calibration import WavelengthCalibration1D
from specreduce.calibration_data import load_MAST_calspec, load_onedstds

from specutils import Spectrum
from specutils.manipulation import median_smooth, gaussian_smooth
from specutils.manipulation import extract_region, gaussian_smooth, convolution_smooth, median_smooth
from specutils.fitting import fit_generic_continuum
from specutils.fitting import fit_generic_continuum, fit_continuum
from specutils.analysis import centroid, fwhm, snr, snr_derived
from specutils.spectra import SpectralRegion
from specutils.manipulation import extract_region
from specutils.fitting import find_lines_derivative
from specutils.fitting import fit_lines



## définition des répertoires et fichiers

In [22]:
# on define frames locations
#CAPTURE_DIR = 'data/20240710_10lac_chcyg_tcrb_m16_m57_betalyr_vega_alphacep/'
#CAPTURE_DIR = 'data/20250822_v1296Aql_altair_Tcrb_rsOph/'
#CAPTURE_DIR='data/20161200_CHATEAU_TIPE/'
#CAPTURE_DIR='data/20250807_deneb_ruchbah_gamcas/'
#CAPTURE_DIR='../../../CAPTURES/20251102_cxdra_v1770cyg_deneb/'
CAPTURE_DIR = '../../../CAPTURES/20260224_m97_merak_procyon/'



OUTPUT_CALIB = 'master_calib.fits'
OUTPUT_SCIENCE = 'master_science.fits'
OUTPUT_RESPONSE = 'response_spec.fits'

# vérifie que ces fichiers existent bien
for _type in (OUTPUT_SCIENCE, OUTPUT_CALIB):
    print("\n",[f.name for f in pathlib.Path(CAPTURE_DIR).glob(_type)])

### Observatory location
# CALC
OBS_LATITUDE = 47.89
OBS_LONGITUDE = -1.50
ALTITUDE: 50

#OHP
#OBS_LATITUDE = 43.931
#OBS_LONGITUDE = 5.712
#ALTITUDE: 600

#MEUSE
#OBS_LONGITUDE = 5.622875
#OBS_LATITUDE = 48.686385
#ALTITUDE = 250

WAVE_RANGE = 3800, 7500



 ['master_science.fits']

 ['master_calib.fits']


In [23]:
%matplotlib widget
import numpy as np
from spectro_dashboard import SpectroDashboard

# 1. Afficher le dashboard
dash = SpectroDashboard()
dash.show()


# trace du spectre

In [26]:
# charge le spectre 2D prétraité
master_science = CCDData.read(CAPTURE_DIR + OUTPUT_SCIENCE, unit=u.Unit('adu'))

# trace le spectre
science_trace = FitTrace(master_science,  
                          bins = 32, 
                          trace_model = models.Polynomial1D(degree=2),
                          peak_method = 'gaussian', 
                          window = 40,
                          #guess=300
                         ) 
#trace_model : one of Chebyshev1D, Legendre1D, Polynomial1D, or Spline1D
#peak_method : One of gaussian, centroid, or max. gaussian

# trace le fond de ciel
bg = Background.two_sided(master_science, 
                          science_trace, 
                          separation = 80, 
                          width = 60) 

# extrait le spectre
extract = BoxcarExtract(master_science, # - bg, 
                        science_trace, 
                        width = 15)

science_spectrum = extract()

# stats sur l'extraction
print(science_trace.trace_model_fit)

# affiche l'extraction
image1_norm = bg.bkg_wimage.astype(float) / bg.bkg_wimage.max()  # Normalise entre 0 et 1
image2_norm = master_science.data.astype(float) / master_science.data.max()

dash.show_image(image1_norm * 0.01 + image2_norm * 0.99, name=master_science.meta['OBJECT'])
dash.ax_img.step(science_spectrum.spectral_axis, science_trace.trace , color='blue', linewidth = '1.0', alpha=1.0, linestyle='dotted')
dash.ax_img.step(science_spectrum.spectral_axis, science_trace.trace + extract.width , color='g', linestyle='dashed', alpha=0.5)
dash.ax_img.step(science_spectrum.spectral_axis, science_trace.trace - extract.width , color='g', linestyle='dashed', alpha=0.5)

# affiche le spectre brut (ADU / px) extrait
dash.clear_spectra()
dash.show_spectrum(science_spectrum.spectral_axis , science_spectrum.flux, label='extraction brut')
dash.ax_spec.set_xlabel('Pixels')
dash.ax_spec.set_ylabel('ADU')
#dash.ax_spec.set_ylim(min(science_spectrum.flux.data), max(science_spectrum.flux.data))
#dash.ax_spec.set_ylim(0, 1e4)



INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
Model: Polynomial1D
Inputs: ('x',)
Outputs: ('y',)
Model set size: 1
Degree: 2
Parameters:
            c0                 c1                    c2          
    ----------------- -------------------- ----------------------
    322.6473794763942 0.007231822438783034 -1.465238515085069e-06
--> Affichage de l'image merak : bin=1, shape=(550, 3856), min=-0, avg=0.0, max=1, std=0.0
--> Spectre 'extraction brut' affiché : 3856 pts, X:[0.0:3855.0]


Text(36.334, 0.5, 'ADU')

# trace du néon de calibration

In [7]:
# charge le spectre 2D prétraité
master_calib = CCDData.read(CAPTURE_DIR + OUTPUT_CALIB, unit=u.Unit('adu'))

# extrait le spectre du néon avec la trace du spectre 
extract = BoxcarExtract(master_calib, science_trace, width = 3)
neon_spectrum = extract()

# affiche le spectre
dash.show_image(master_calib, name='calib')
dash.clear_spectra()
dash.show_spectrum(neon_spectrum.spectral_axis , neon_spectrum.flux, label='extraction')
dash.ax_spec.set_xlabel('Pixels')
dash.ax_spec.set_ylabel('ADU')
#dash.ax_spec.axis("off")
#dash.cbar.remove()
dash.ax_spec.set_ylim(min(neon_spectrum.flux.data), max(neon_spectrum.flux.data))
#dash.ax_spec.set_ylim(0, 1e6)
#dash.ax_spec.relim()   #set_ylim(bottom=None, top=None)
#dash.ax_spec.autoscale_view()



### recherche les raies 
lines = find_lines_derivative(neon_spectrum, flux_threshold=10000)
for l in lines:
    print(f"{l['line_type']} : {l['line_center']}") 
    #print('\nabsorption: \n', l['line_type'] == 'absorption') 

# readjust x-pixel positions accordingly
#PIXELS = [78.099, 128.005, 200.651, 339.185, 468.038, 550.164, 617.539, 677.917, 796.963]*u.pix

INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
--> Affichage de l'image calib : bin=1, shape=(550, 3856), min=-96, avg=5659.1, max=64944, std=15363.0
--> Spectre 'extraction' affiché : 3856 pts, X:[0.0:3855.0]
emission : 1540.0 pix
emission : 1550.0 pix
emission : 1601.0 pix
emission : 1905.0 pix
emission : 1919.0 pix
emission : 1952.0 pix
emission : 1967.0 pix
emission : 1987.0 pix
emission : 1993.0 pix
emission : 1997.0 pix
emission : 2001.0 pix
emission : 2017.0 pix
emission : 2039.0 pix
emission : 2050.0 pix
emission : 2069.0 pix
emission : 2098.0 pix
emission : 2109.0 pix
emission : 2146.0 pix
emission : 2162.0 pix
emission : 2182.0 pix
emission : 2186.0 pix
emission : 2201.0 pix
emission : 2240.0 pix
emission : 2247.0 pix
emission : 2259.0 pix
emission : 2276.0 pix
emission : 2305.0 pix
emission : 2309.0 pix
emission : 2334.0 pix
emission : 2346.0 pix
emission : 2354.0 pix
emission : 2381.0 pix
emission : 2385

# calibration du néon

In [8]:
### alpy-600 + neon builtin
#pixels = [351, 589, 1045, 1403, 1885, 2073]*u.pix
#wavelength = [4200.67, 4764.87, 5852.49, 6677.28, 7272.94, 7635.11]*u.AA
#pixels =     [355,     1045,    1171     , 1284,   1406,      1535  , 1674  , 1840 ]*u.pix
#wavelength = [4200.67, 5852.49, 6143.06,  6402.25  , 6677.28  ,   6965.43 , 7272.94 , 7635.11]*u.AA
#wavelength_ = wavelength #= [4200.67, 5852.49, 6143.06,  6402.25  , 6677.28  ,   6965.43 , 7272.94 , 7635.11]

# Dados200 + Xenon lamp
#pixels = [191, 400, 871, 1105, 1533]*u.pix
#pixels = [188, 396, 868, 1102, 1530]*u.pix
#wavelength = [4671.22, 5028.28, 5852.49, 6266.49, 7031.41]*u.AA

#wavelength =[4671.22, 5028.28, 5852.49, 6266.49, 7031.41]*u.AA
#pixels = [191, 401, 871, 1105, 1533]*u.pix

### Dados200 + neon lamp
#pixels = [868, 1276, 2342, 3635, 4263]*u.pix
#wavelength = [6506.53, 6532.88, 6598.95, 6678.28, 6717.04]*u.AA

### StarEx-2400 + neon lamp
#pixels = [868, 1276, 2342, 3635, 4263]*u.pix
#pixels = [853, 1266, 2314, 3599, 4238]*u.pix
#wavelength = [6506.53, 6532.88, 6598.95, 6678.28, 6717.04]*u.AA
#wavelength = WAVELENGTH
#pixels = PIXELS

#line_list = QTable([pixels, wavelength], names=["pixel_center", "wavelength"])
#input_spectrum, matched_line_list=None, line_pixels=None, line_wavelengths=None, catalog=None, input_model=Linear1D(), fitter=None

# neon lines - DADOS 200 + PO Uranus
pixels = [655, 1601, 2556, 3298, 3767] * u.pix
wavelength = [4333.42, 5400.56, 6506.52, 7383.98, 7948.18]* u.AA

# balmer lines - DADOS 200 + PO Uranus
#wavelength = [3889.05, 3970.08, 4101.75, 4340.48, 4861.34, 6562.81] * u.AA
#pixels = [412, 480, 604, 820, 1293, 2754] * u.pix
#pixels = [414, 487, 607, 821, 1284, 2754] * u.pix

#STD positions: 
#line_pos: [62, 681, 937, 1306] 
#pixels = [1362, 1982, 2230, 2612]  * u.pix
#pixels = [1222, 1840, 2090, 2472]  * u.pix
#pixels = [901, 1838, 2818, 3235]  * u.pix

#wavelength = [4333.42, 5400.56, 5852.48, 6506.52] * u.AA
#wavelength = [4333.56, 5400.56, 6532.88, 7032.41] * u.AA

calibration = WavelengthCalibration1D(input_spectrum = neon_spectrum,
      #matched_line_list = line_list,
      line_wavelengths = wavelength,
      line_pixels = pixels,
      input_model = models.Polynomial1D(degree = 2),
      #fitter = fitting.LMLSQFitter()
      #fitter = fitting.LinearLSQFitter()
     )

print('residuals :', calibration.residuals )
print('fitted ', calibration.fitted_model )

neon_calibrated_spectrum = calibration.apply_to_spectrum(neon_spectrum)

dash.clear_spectra()
dash.show_spectrum(neon_calibrated_spectrum.spectral_axis, neon_calibrated_spectrum.flux, label='calibrated')
dash.ax_spec.set_xlabel('Lambda (angstrom)')
dash.ax_spec.set_ylabel('ADU')
#dash.ax_spec.set_ylim(0, 0.1e5)

for line in wavelength:
    dash.ax_spec.axvline(line.value, 0.95, 1.0, color = 'b', lw = 1.0)
    dash.ax_spec.axvline(line.value, color = 'b', lw = 1.0, linestyle = ':')

#dash.ax_spec.set_ylim(min(neon_calibrated_spectrum.flux.data), max(neon_calibrated_spectrum.flux.data))



residuals : [ 0.05479358 -0.21107535  0.36188043 -0.32316762  0.11756896] Angstrom
fitted  Model: Polynomial1D
Inputs: ('x',)
Outputs: ('y',)
Model set size: 1
Degree: 2
Parameters:
            c0                c1                   c2          
         Angstrom       Angstrom / pix      Angstrom / pix2    
    ----------------- ------------------ ----------------------
    3610.378241038091 1.0937574661200626 1.5327370367369153e-05
--> Spectre 'calibrated' affiché : 3856 pts, X:[3610.4:8054.6]


# calibration du spectre science

In [9]:

science_calibrated_spectrum = calibration.apply_to_spectrum(science_spectrum)

print ("calibrated spectrum:", min(science_calibrated_spectrum.flux), max(science_calibrated_spectrum.flux))

dash.clear_spectra()
dash.show_spectrum(science_calibrated_spectrum.spectral_axis, science_calibrated_spectrum.flux, label='calibrated')
dash.ax_spec.set_xlabel('Lambda (angstrom)')
dash.ax_spec.set_ylabel('ADU')
#dash.ax_spec.set_ylim(min(science_calibrated_spectrum.flux.data), max(science_calibrated_spectrum.flux.data))
#dash.ax_spec.relim()   #set_ylim(bottom=None, top=None)
#dash.ax_spec.autoscale_view()


calibrated spectrum: 3351.093523425325 adu 15410650.52283717 adu
--> Spectre 'calibrated' affiché : 3856 pts, X:[3610.4:8054.6]


Text(36.334, 0.5, 'ADU')

# lissage (optionel)

In [10]:
smooth_spec = median_smooth(science_calibrated_spectrum, width = 3) 

#plt.step(science_calibrated_spectrum.wavelength, science_calibrated_spectrum.flux + 100*u.adu, color = 'grey', linewidth = '0.6', label = 'orig')
#plt.step(smooth_spec.wavelength, smooth_spec.flux , color = 'red', linewidth = '0.6', label = 'smoothed')
#plt.legend(loc=('best'))

#dash.clear_spectra()
#dash.show_spectrum(science_calibrated_spectrum.spectral_axis, science_calibrated_spectrum.flux, label='calibrated')
#dash.show_spectrum(smooth_spec.spectral_axis, smooth_spec.flux, label='median_smooth')

#dash.ax_spec.set_ylim(min(science_calibrated_spectrum.flux.data), max(science_calibrated_spectrum.flux.data))

### decide to keep the median smoothed version ?
science_calibrated_spectrum = smooth_spec
print ("smoothed spectrum:", min(science_calibrated_spectrum.flux), max(science_calibrated_spectrum.flux))




smoothed spectrum: 4547.510047437248 adu 14760525.627808494 adu


# masse d'air et distance zenitale

In [11]:
### compute zenith distance (if not present in fit header)
TARGET = master_science.meta['OBJECT']
target_coord = SkyCoord.from_name(TARGET)
target_time = Time(master_science.meta['DATE-OBS'])
obs_coord = EarthLocation(lon = OBS_LONGITUDE * u.deg, lat = OBS_LATITUDE * u.deg)
altaz = AltAz(obstime=target_time, location = obs_coord)

ZD = target_coord.transform_to(AltAz(obstime = target_time, location = obs_coord)).zen
airmass = 1.0 / np.cos(ZD)
print(f'computed ZD={ZD}, airmass={airmass}')

computed ZD=45.41832799875659 deg, airmass=1.4246531208562736


# réponse instrumentale

In [12]:
###
### version externe : utilisation de la réponse instrumentale calculée par SpecInti
###
from astropy.nddata import NDDataRef
from specutils.manipulation import FluxConservingResampler

final_spec: Spectrum = None
respFile = CAPTURE_DIR + '_rep_20260224.fits'


if len(fits.open(respFile)) == 1:
    # standard FITS spectrum with data in hdu0
    print("standard FIT response")
    resp1d: Spectrum = Spectrum.read(respFile)

elif len(fits.open(respFile)) == 2:
    # data are in hdu1
    print("multiple hdus FIT response")
    _spc = CCDData.read(respFile, hdu=1, unit=u.Unit('adu'))
    resp1d = Spectrum(spectral_axis = _spc.data['wavelength'] * u.Unit('Angstrom'), 
                        flux = _spc.data['flux'] * u.Unit('mJy')
                        )
else:
    print(f"{respFile} : no data found in HDUs")

print ("raw reponseFile:", min(resp1d.flux), max(resp1d.flux))
print ("calibrated spectrum:", min(science_calibrated_spectrum.flux), max(science_calibrated_spectrum.flux))

#fcal = FluxCalibration() 
#fcal(airmass=airmass, object_spectrum=science_calibrated_spectrum)
#final_spec = fcal.apply_sensfunc(resp1d)
#print ("final spectrum(sensfunc):", min(final_spec.flux), max(final_spec.flux))

_factor = round(resp1d.shape[0] / science_calibrated_spectrum.shape[0])
print(f"{science_calibrated_spectrum.shape[0]=}, {resp1d.shape[0]=}, {_factor=:.6f}")

resampler = FluxConservingResampler(extrapolation_treatment='truncate')
resp_resampled = resampler(resp1d, science_calibrated_spectrum.spectral_axis)
print ("resampled reponseFile:", min(resp_resampled.flux), max(resp_resampled.flux))
print(f"{resp_resampled.shape[0]=}")

spec_resampled = resampler(science_calibrated_spectrum, resp_resampled.spectral_axis)
print ("resampled spectrum:", min(resp_resampled.flux), max(resp_resampled.flux))
print(f"{spec_resampled.shape[0]=}")


final_spec = spec_resampled / resp_resampled
print ("final spectrum:", min(final_spec.flux), max(final_spec.flux))

print('response applied')

dash.clear_spectra()
dash.show_spectrum(final_spec.spectral_axis, final_spec.flux, label='response applied')
dash.ax_spec.set_ylim(min(final_spec.flux.data), max(final_spec.flux.data))


standard FIT response
raw reponseFile: 0.06357097625732422 6.8541107177734375
calibrated spectrum: 4547.510047437248 adu 14760525.627808494 adu
science_calibrated_spectrum.shape[0]=3856, resp1d.shape[0]=6401, _factor=2.000000
resampled reponseFile: 0.06357278842821774 6.854015097114184
resp_resampled.shape[0]=3226
resampled spectrum: 0.06357278842821774 6.854015097114184
spec_resampled.shape[0]=3226
final spectrum: 274625.6720338632 adu 2211411.1979723955 adu
response applied
--> Spectre 'response applied' affiché : 3226 pts, X:[3610.4:7297.2]


(274625.6720338632, 2211411.1979723955)

# cropping, normalisation et sauvegarde

In [14]:
#
# on normalize à 1 autour de 6000 AA

sci_mean_norm_region = final_spec[4950 * u.AA: 5000 * u.AA].flux.mean()        # DADOS200 : low resolution
#sci_mean_norm_region = final_spec[6500 * u.AA: 6520 * u.AA].flux.mean()       # starEx2400 : high resolution

#final_spec = np.nan_to_num(final_spec, nan=1)
lambda_crop_min = 3700*u.AA
lambda_crop_max = 7100*u.AA

print(f"{sci_mean_norm_region=}")
_spec = Spectrum(spectral_axis = final_spec[lambda_crop_min:lambda_crop_max].wavelength, flux = final_spec[lambda_crop_min:lambda_crop_max].flux / sci_mean_norm_region)  

# n applique un décalage si besoin
#SHIFT_WAVE = 0 * u.AA
#_spec = Spectrum(spectral_axis = _spec.wavelength + SHIFT_WAVE, flux = _spec.flux)  

# on sauve le spectre eu format FITS
_spec.write(CAPTURE_DIR + 'spectre-1D.fit', overwrite=True) #, format='tabular-fits')


### on sauve au format ASCII pour fityk ou Excel
np.savetxt(CAPTURE_DIR + 'spectre-1D-tabular.dat', 
           np.array([_spec.spectral_axis.value, _spec.flux.value]).T, 
           fmt='%.8f', 
           header='WAVELENGTH FLUX')

dash.clear_spectra()
dash.show_spectrum(_spec.spectral_axis, _spec.flux, label='response applied')
dash.ax_spec.set_ylim(min(_spec.flux.data), max(_spec.flux.data))


sci_mean_norm_region=<Quantity 1418255.73578155 adu>
--> Spectre 'response applied' affiché : 2978 pts, X:[3700.2:7099.6]


(0.25945768300973243, 1.5592471387071538)

# affichage final

In [15]:
#%matplotlib widget
#import numpy as np
#from spectro_dashboard import SpectroDashboard

#dash = SpectroDashboard()
#dash.show()

spc = Spectrum.read(CAPTURE_DIR + 'spectre-1D-tabular.dat', ignore_missing_simple=True, format='ASCII',  
                    column_mapping = {'FLUX': ('flux', 'adu'), 'WAVELENGTH': ('spectral_axis', 'AA')})

#specinti_spec = Spectrum.read(CAPTURE_DIR + '_ruchbah_20250807_916.fit')
#specinti_spec = Spectrum.read(CAPTURE_DIR + '_gamcas_20250807_912.fit')
#specinti_spec = Spectrum.read(CAPTURE_DIR + '_lamcyg_20251102_914.fits')
#specinti_spec = Spectrum.read(CAPTURE_DIR + '_v1770cyg_20251102_906.fits')

lambda_crop_min = 3800*u.AA
lambda_crop_max = 7300*u.AA

print (min(spc.flux), max(spc.flux))

dash.clear_spectra()
dash.show_image(master_science, name=master_science.meta['OBJECT'])
dash.show_spectrum(spc[lambda_crop_min:lambda_crop_max].wavelength, spc[lambda_crop_min:lambda_crop_max].flux, label="final")
#dash.show_spectrum(specinti_spec[lambda_crop_min:lambda_crop_max].wavelength, specinti_spec[lambda_crop_min:lambda_crop_max].flux, label="specinti")
dash.ax_spec.set_ylim(min(spc.flux.data), max(spc.flux.data))

#dash.ax_spec.set_ylim(min(spc.flux), max(spc.flux))

#dash.show_spectrum(specinti_spec[3800*u.AA:7300*u.AA].wavelength, specinti_spec[3800*u.AA:7300*u.AA].flux, label="specinti")


0.25945768 adu 1.55924714 adu
--> Affichage de l'image merak : bin=1, shape=(550, 3856), min=-334270, avg=3438.4, max=1820703, std=31609.5
--> Spectre 'final' affiché : 2887 pts, X:[3800.1:7099.6]


(0.25945768, 1.55924714)